# 06 · User Persona Generator

The **User Persona** service creates a structured, research-grounded user persona from a
product/service description. The persona is a design tool — a semi-fictional representation of
a key user segment based on UX research and product strategy.

## What Makes a Good Persona?

- **Specificity**: Name, age, job, life situation — real enough to empathize with
- **Goals**: What the user is trying to achieve (not product-specific)
- **Pain points**: Frustrations that the product should solve
- **Motivations**: What drives behaviour and decision-making
- **Technology comfort**: Adoption curve and device preferences
- **Behavioural traits**: How they actually use products day-to-day

## Key Design Decisions

| Decision | Rationale |
|---|---|
| Structured JSON output | Enables frontend to render components; easy to store and compare personas |
| Gender prompt repeated 4× | Prevents LLM drift back to default gender after initial instruction |
| `temperature=0.7` | Higher than other services — creativity matters for realistic persona detail |
| No few-shot examples | Keeps prompt concise; the schema alone guides output structure reliably |

In [1]:
%pip install -q langchain-groq python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os, json
from pathlib import Path
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage

load_dotenv(dotenv_path=Path('..') / '.env')

# Higher temperature for more varied, creative personas
llm = ChatGroq(model='llama-3.3-70b-versatile', temperature=0.7)
print('Ready')

Ready


## The Persona Schema

We enforce a strict schema via the system prompt. The LLM must output valid JSON matching this structure.

In [3]:
PERSONA_SCHEMA = {
    'name': 'string — first and last name',
    'age': 'integer',
    'gender': 'string (ENFORCED by caller)',
    'occupation': 'string',
    'location': 'city, country',
    'income_bracket': 'low / lower-middle / middle / upper-middle / high',
    'education': 'highest education level',
    'bio': '2-3 sentence life/work situation',
    'goals': ['...', '...', '...'],
    'pain_points': ['...', '...', '...'],
    'motivations': ['...', '...'],
    'tech_comfort': 'low / medium / high / power-user',
    'devices': ['...'],
    'apps_used': ['...'],
    'quote': 'first-person quote that captures their mindset',
    'buying_behaviour': 'description of decision-making process',
    'frustrations': ['...'],
    'preferred_channels': ['...'],
}

print('Schema keys:', list(PERSONA_SCHEMA.keys()))

Schema keys: ['name', 'age', 'gender', 'occupation', 'location', 'income_bracket', 'education', 'bio', 'goals', 'pain_points', 'motivations', 'tech_comfort', 'devices', 'apps_used', 'quote', 'buying_behaviour', 'frustrations', 'preferred_channels']


## Prompt Engineering: Gender Enforcement

LLMs tend to default to common name+gender combinations. Repeating the gender instruction
4 times (opening, before name, before bio, before quote) prevents drift.

In [4]:
def build_persona_prompt(gender: str) -> str:
    g = gender.lower()
    return f"""
You are a UX research expert creating a detailed user persona.
CRITICAL: This persona MUST be {g}. Do not change the gender.

Generate a {g} persona that strictly matches this JSON schema:

{json.dumps(PERSONA_SCHEMA, indent=2)}

IMPORTANT RULES:
1. The persona MUST be {g}
2. Use a name appropriate for {g} gender
3. All lists must have 3–5 items
4. Bio should be 2–3 sentences describing a realistic {g} user
5. Quote should be realistic and personal — this person is {g}

Return ONLY valid JSON. No markdown, no explanation.
""".strip()

print('Prompt for gender=female:')
print(build_persona_prompt('female')[:500])

Prompt for gender=female:
You are a UX research expert creating a detailed user persona.
CRITICAL: This persona MUST be female. Do not change the gender.

Generate a female persona that strictly matches this JSON schema:

{
  "name": "string \u2014 first and last name",
  "age": "integer",
  "gender": "string (ENFORCED by caller)",
  "occupation": "string",
  "location": "city, country",
  "income_bracket": "low / lower-middle / middle / upper-middle / high",
  "education": "highest education level",
  "bio": "2-3 senten


## Generate a Persona

In [5]:
async def generate_persona(product_description: str, gender: str = 'female') -> dict:
    """Generate a structured user persona for a given product and gender."""
    system_prompt = build_persona_prompt(gender)

    user_content = (
        f'Create a {gender} persona for the following product/service:\n\n'
        f'{product_description}\n\n'
        f'Remember: the persona must be {gender}.'
    )

    msg = await llm.ainvoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_content),
    ])

    raw = msg.content.strip()
    if raw.startswith('```'):
        raw = raw.split('```')[1].lstrip('json').strip()

    try:
        persona = json.loads(raw)
        return persona
    except json.JSONDecodeError as e:
        print(f'JSON parse error: {e}')
        return {'raw': raw}


product = (
    'ZenFlow is a mindfulness and meditation app for busy professionals aged 28–45. '
    'It provides guided meditations, mood tracking, stress management tools, '
    'and integrates with Apple Watch and Fitbit. Subscription-based, $9.99/month.'
)

persona = await generate_persona(product, gender='female')
print('Persona generated!')
print(f'Name: {persona.get("name")}')
print(f'Age: {persona.get("age")}')
print(f'Gender: {persona.get("gender")}')
print(f'Occupation: {persona.get("occupation")}')

Persona generated!
Name: Emily Chen
Age: 32
Gender: female
Occupation: Marketing Manager


## Full Persona Display

In [6]:
from IPython.display import Markdown

def render_persona_md(p: dict) -> str:
    lines = [
        f'# {p.get("name", "?")} — User Persona',
        f'',
        f'> *"{p.get("quote", "")}"*',
        f'',
        f'**Age:** {p.get("age", "?")} | **Gender:** {p.get("gender", "?")} | **Location:** {p.get("location", "?")}',
        f'**Occupation:** {p.get("occupation", "?")} | **Income:** {p.get("income_bracket", "?")} | **Tech comfort:** {p.get("tech_comfort", "?")}',
        f'',
        f'## Bio',
        p.get('bio', ''),
        f'',
        f'## Goals',
        '\n'.join(f'- {g}' for g in p.get('goals', [])),
        f'',
        f'## Pain Points',
        '\n'.join(f'- {pp}' for pp in p.get('pain_points', [])),
        f'',
        f'## Motivations',
        '\n'.join(f'- {m}' for m in p.get('motivations', [])),
        f'',
        f'## Frustrations',
        '\n'.join(f'- {f}' for f in p.get('frustrations', [])),
        f'',
        f'## Devices & Apps',
        f'**Devices:** {", ".join(p.get("devices", []))}',
        f'**Apps used:** {", ".join(p.get("apps_used", []))}',
    ]
    return '\n'.join(lines)

display(Markdown(render_persona_md(persona)))

# Emily Chen — User Persona

> *"I feel like I'm constantly running on a treadmill and getting nowhere, I just want to find a way to slow down and breathe without feeling guilty about it."*

**Age:** 32 | **Gender:** female | **Location:** New York, USA
**Occupation:** Marketing Manager | **Income:** upper-middle | **Tech comfort:** high

## Bio
Emily is a busy marketing manager who often finds herself stressed and overwhelmed with work and personal life. She tries to maintain a healthy work-life balance, but it's a constant struggle. She's recently started prioritizing self-care and mindfulness to improve her mental well-being.

## Goals
- Reduce stress and anxiety
- Improve sleep quality
- Increase productivity and focus

## Pain Points
- Difficulty finding time for self-care
- Struggling to quiet her mind and relax
- Feeling guilty for taking time for herself

## Motivations
- Desire to feel more grounded and centered
- Wanting to be a positive role model for her loved ones

## Frustrations
- Too many meditation apps feel generic and not tailored to her needs
- Difficulty tracking progress and staying motivated
- Feeling like she's not doing it 'right'

## Devices & Apps
**Devices:** iPhone 13, Apple Watch Series 7, MacBook Air
**Apps used:** Headspace, Calm, Fitbit Coach

## Generate Multiple Personas

Best practice: create at least 2–3 personas with different demographics for the same product.

In [7]:
import asyncio

# Generate male and non-binary personas in parallel
results = await asyncio.gather(
    generate_persona(product, gender='male'),
    generate_persona(product, gender='non-binary'),
)

for p in results:
    print(f'--- {p.get("name")} ({p.get("gender")}, {p.get("age")}) ---')
    print(f'  Occupation: {p.get("occupation")}')
    print(f'  Top goal: {p.get("goals", ["?"])[0]}')
    print(f'  Quote: "{p.get("quote", "")}"')
    print()

--- Ethan Thompson (male, 37) ---
  Occupation: Marketing Manager
  Top goal: Reduce stress and anxiety
  Quote: "I feel like I'm constantly running on a treadmill and can't catch my breath. I need something to help me slow down and find some peace in the chaos."

--- Rowan Sawyer (non-binary, 32) ---
  Occupation: Marketing Manager
  Top goal: Reduce stress and anxiety
  Quote: "I'm tired of feeling like I'm just going through the motions – I want to be more present and mindful in my daily life, and I'm hoping that meditation and mindfulness can help me get there."



## Gender Enforcement Validation

We verify that the LLM actually respects the gender instruction across multiple requests.

In [8]:
# Run 3 female personas and check gender compliance
genders_seen = []
for _ in range(3):
    p = await generate_persona(product, gender='female')
    genders_seen.append(p.get('gender', 'unknown').lower())

print('Generated genders:', genders_seen)
female_match = sum(1 for g in genders_seen if 'female' in g or 'woman' in g)
print(f'Correct gender: {female_match}/3 ({female_match/3*100:.0f}%)')

Generated genders: ['female', 'female', 'female']
Correct gender: 3/3 (100%)


## Summary

- Gender enforcement via **instruction repetition** (4 positions in the prompt) achieves near-100% compliance.
- The structured JSON schema makes persona data **component-ready** for frontend rendering.
- Parallel generation of multi-gender personas is safe — each call is fully stateless.
- `temperature=0.7` balances persona realism with variety — lower values produce formulaic outputs.